<a href="https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Spark/6-Performance_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color:#E25822; font-size:2.5em;">Spark Performance Optimization</h1>

**Course:** Big Data Analytics &nbsp;|&nbsp; **Module 5 – Introduction to Spark**

---

By the end of this notebook you will be able to:

- Explain the Spark execution model (Jobs → Stages → Tasks) and understand the DAG
- Decide when and how to cache or persist DataFrames
- Tune partition counts to maximise parallelism
- Accelerate joins with broadcast variables
- Read and interpret Catalyst query plans
- Benchmark PySpark against pandas and know which tool to choose
- Apply a checklist of best practices to every Spark job

<h2 style="color:#E25822;">Why Performance Matters in Distributed Computing</h2>

Spark runs on a **cluster** of machines.  Every time data crosses the network (a *shuffle*) or a computation is re-run unnecessarily, you pay in:

| Cost dimension | What it means |
|---|---|
| **Time** | Users wait longer; interactive jobs feel sluggish |
| **Money** | Cloud clusters bill by the second — a 10× slower job costs 10× more |
| **Resources** | Wasted memory/CPU prevents other users from running jobs |

### When does Spark shine vs. pandas?

```
Data size        Recommended tool
──────────────────────────────────────────────────────
< 1 GB           pandas  (single machine, simpler API)
1 GB – 10 GB     Either – benchmark first
> 10 GB          PySpark (distributed, scales out)
```

**Key insight:** Spark has overhead — spinning up a cluster, serialising tasks, scheduling stages.  For small data this overhead *dominates*, and pandas wins.  The crossover point is roughly 1–5 GB depending on the workload.  We will measure this later in the notebook.

> **Rule of thumb:** If your data fits comfortably in a single machine's RAM, start with pandas.  Graduate to Spark when you need horizontal scale.

<h2 style="color:#E25822;">Setup – Install PySpark and Create a SparkSession</h2>

Run the cell below once per Colab session.  It installs PySpark and creates a local SparkSession sized for the Colab free tier (2 vCPUs, ~12 GB RAM).

In [ ]:
# ── Install PySpark (skip if already installed) ────────────────────────────
import importlib, subprocess, sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark", "-q"])
    print("PySpark installed.")
else:
    print("PySpark already available.")

# ── Imports ────────────────────────────────────────────────────────────────
import time
import random
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType
)
from pyspark.sql.functions import broadcast
from pyspark.storagelevel import StorageLevel

# ── Create SparkSession ────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .master("local[*]")                        # use all available cores
    .appName("SparkPerformanceOptimization")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")  # tune for local mode
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")  # suppress verbose INFO logs

print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")
print(f"Available cores: {spark.sparkContext.defaultParallelism}")

<h2 style="color:#E25822;">Part 1 – Understanding Spark Execution</h2>

### 1.1 Jobs → Stages → Tasks

Every time you call an **action** (e.g. `count()`, `show()`, `write()`), Spark creates a **Job**.  Each Job is split into **Stages** at shuffle boundaries.  Each Stage is divided into **Tasks** that run in parallel on cluster nodes.

```
┌─────────────────────────────────────────────┐
│                   JOB                        │
│  (triggered by one action, e.g. count())     │
│                                             │
│  ┌──────────────────┐  ┌──────────────────┐ │
│  │     STAGE 1      │  │     STAGE 2      │ │
│  │  (before shuffle)│  │  (after shuffle) │ │
│  │                  │  │                  │ │
│  │ Task Task Task   │  │ Task Task Task   │ │
│  │  [p1] [p2] [p3]  │  │  [p1] [p2] [p3] │ │
│  └──────────────────┘  └──────────────────┘ │
│              ↕ shuffle / network             │
└─────────────────────────────────────────────┘
```

Each **Task** processes exactly one **partition** of data.  More partitions = more parallelism (up to the number of available CPU cores).

---

### 1.2 The DAG – Directed Acyclic Graph

Spark does not execute transformations immediately — it builds a **DAG** of all transformations and only executes when an action is called (**lazy evaluation**).

```
read CSV  →  filter  →  groupBy  →  agg  →  write
   ↓            ↓           ↓         ↓        ↓
 RDD/DF      RDD/DF      SHUFFLE   RDD/DF    ACTION
                           ↑
                    Stage boundary!
```

Benefits of lazy evaluation:
- Spark can **optimise the whole pipeline** before running anything (Catalyst optimiser)
- Unreferenced columns are pruned automatically
- Filters are pushed as early as possible

---

### 1.3 Narrow vs Wide Transformations

```
NARROW TRANSFORMATION              WIDE TRANSFORMATION
(no shuffle – stays in partition)  (requires shuffle across network)

Partition 1 ──→ Partition 1        Partition 1 ─┐
Partition 2 ──→ Partition 2        Partition 2 ─┼──→ Partition A
Partition 3 ──→ Partition 3        Partition 3 ─┘
                                   Partition 1 ─┐
Examples:                          Partition 2 ─┼──→ Partition B
  map(), filter(),                 Partition 3 ─┘
  select(), withColumn()           Examples:
                                     groupBy(), join(),
                                     distinct(), orderBy()
```

| Transformation type | Network I/O | Stage boundary | Cost |
|---|---|---|---|
| **Narrow** | None | No | Low |
| **Wide** | High (shuffle) | Yes | High |

**Minimising shuffles** is the single most impactful performance optimisation.

---

### 1.4 Why Shuffles Are Expensive

During a shuffle:
1. Each task **writes** its output to local disk (spill)
2. The data is **transferred over the network** to the correct node
3. The receiving task **reads** the data and sorts/merges it

Disk I/O + network I/O + sort = 3× the pain of a narrow transformation.

In [ ]:
# ── Demonstrating narrow vs wide transformations via explain() ─────────────
df_demo = spark.range(1_000_000).toDF("id")  \
    .withColumn("value", F.rand(seed=42) * 100) \
    .withColumn("category", (F.col("id") % 5).cast(StringType()))

print("=" * 60)
print("NARROW: filter + select  (no shuffle)")
print("=" * 60)
narrow_df = df_demo.filter(F.col("value") > 50).select("id", "value")
narrow_df.explain()

print("=" * 60)
print("WIDE: groupBy + agg  (shuffle required)")
print("=" * 60)
wide_df = df_demo.groupBy("category").agg(F.avg("value").alias("avg_value"))
wide_df.explain()

<h2 style="color:#E25822;">Part 2 – Caching and Persistence</h2>

### 2.1 When to Cache

Cache a DataFrame when you will **use it more than once** and it is **expensive to recompute**.  Without caching, Spark re-reads and re-transforms the data from scratch every time an action is called.

```
Without cache:
  action 1 → read file → filter → groupBy → result 1
  action 2 → read file → filter → groupBy → result 2   ← repeated work!

With cache:
  action 1 → read file → filter → groupBy → cache → result 1
  action 2 →                              ↑ cache hit → result 2  ✓
```

### 2.2 cache() vs persist()

| Method | Storage level | When to use |
|---|---|---|
| `df.cache()` | MEMORY_AND_DISK (default) | Quick shorthand, medium datasets |
| `df.persist()` | MEMORY_AND_DISK (default) | Same as cache() |
| `df.persist(StorageLevel.MEMORY_ONLY)` | RAM only | Fast access, fits in RAM |
| `df.persist(StorageLevel.DISK_ONLY)` | Disk only | Very large datasets, RAM scarce |
| `df.persist(StorageLevel.OFF_HEAP)` | Off-heap RAM | Reduce GC pressure |

`cache()` is simply an alias for `persist()` with the default storage level.

### 2.3 Storage Levels Reference

| Storage Level | RAM | Disk | Serialised | Replicated | Best for |
|---|:---:|:---:|:---:|:---:|---|
| MEMORY_ONLY | ✓ | ✗ | ✗ | ✗ | Small DataFrames, fast reuse |
| MEMORY_AND_DISK | ✓ | ✓ | ✗ | ✗ | General purpose (default) |
| MEMORY_ONLY_SER | ✓ | ✗ | ✓ | ✗ | Save memory, slower CPU |
| MEMORY_AND_DISK_SER | ✓ | ✓ | ✓ | ✗ | Balanced memory/speed |
| DISK_ONLY | ✗ | ✓ | ✓ | ✗ | Very large, RAM constrained |
| MEMORY_AND_DISK_2 | ✓ | ✓ | ✗ | ✓ | Fault-tolerant clusters |

In [ ]:
# ── Timed example: without cache vs with cache ─────────────────────────────
# Generate a DataFrame with 5 million rows
NUM_ROWS = 5_000_000

def make_large_df(n=NUM_ROWS):
    """Create a synthetic DataFrame with n rows."""
    return (
        spark.range(n)
        .withColumn("value",    F.rand(seed=7) * 1000)
        .withColumn("score",    F.randn(seed=13) * 50 + 100)
        .withColumn("category", (F.col("id") % 10).cast(StringType()))
        .withColumn("flag",     (F.rand(seed=3) > 0.5).cast("integer"))
    )

# ── WITHOUT cache ──────────────────────────────────────────────────────────
print(f"Generating DataFrame with {NUM_ROWS:,} rows ...")
df_no_cache = make_large_df()

t0 = time.time()
agg1 = df_no_cache.groupBy("category").agg(F.avg("value").alias("avg_val")).count()
agg2 = df_no_cache.groupBy("category").agg(F.sum("score").alias("sum_score")).count()
agg3 = df_no_cache.groupBy("flag").agg(F.count("id").alias("cnt")).count()
time_no_cache = time.time() - t0

print(f"\nWithout cache – 3 aggregations: {time_no_cache:.2f}s")

# ── WITH cache ─────────────────────────────────────────────────────────────
df_cached = make_large_df()
df_cached.cache()                 # marks for caching

# IMPORTANT: cache is populated lazily – the first action fills it
t_populate = time.time()
_ = df_cached.count()             # triggers cache population
populate_time = time.time() - t_populate
print(f"Cache population (count): {populate_time:.2f}s")

t1 = time.time()
agg1c = df_cached.groupBy("category").agg(F.avg("value").alias("avg_val")).count()
agg2c = df_cached.groupBy("category").agg(F.sum("score").alias("sum_score")).count()
agg3c = df_cached.groupBy("flag").agg(F.count("id").alias("cnt")).count()
time_with_cache = time.time() - t1

print(f"With cache    – 3 aggregations: {time_with_cache:.2f}s")
speedup = time_no_cache / time_with_cache if time_with_cache > 0 else float('inf')
print(f"\nSpeedup from caching: {speedup:.1f}×")
print("\n(Expected: 2–5× speedup for repeated aggregations on cached data)")

In [ ]:
# ── Memory management: always unpersist when done ─────────────────────────
df_cached.unpersist()
print("DataFrame unpersisted – memory released.")

# persist() with explicit storage level
df_persist_demo = make_large_df()
df_persist_demo.persist(StorageLevel.MEMORY_AND_DISK)
_ = df_persist_demo.count()   # populate
print("Persisted with MEMORY_AND_DISK.")
df_persist_demo.unpersist()
print("Unpersisted.")

# Show storage level programmatically
df_check = make_large_df()
df_check.cache()
print(f"\nStorage level after cache(): {df_check.storageLevel}")
df_check.unpersist()
print(f"Storage level after unpersist(): {df_check.storageLevel}")

<h2 style="color:#E25822;">Part 3 – Partitioning</h2>

### 3.1 What Is a Partition?

A **partition** is a contiguous chunk of data stored on one node and processed by one Task.  Think of it as a page in a book — the book is your DataFrame, each page is a partition.

```
DataFrame (100M rows)
├── Partition 0  (12.5M rows)  →  Worker node 1, Core 0
├── Partition 1  (12.5M rows)  →  Worker node 1, Core 1
├── Partition 2  (12.5M rows)  →  Worker node 2, Core 0
├── Partition 3  (12.5M rows)  →  Worker node 2, Core 1
│   ...                           ...
└── Partition 7  (12.5M rows)  →  Worker node 4, Core 1
```

**Rule of thumb:** aim for 2–4 partitions per CPU core, each partition 128 MB–512 MB.

### 3.2 repartition() vs coalesce()

| Feature | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | Yes (full shuffle) | No (narrow, merges partitions) |
| Direction | Increase **or** decrease | Decrease only |
| Data balance | Perfectly balanced | May be uneven |
| Cost | Higher | Lower |
| Use when | Need more partitions or even split | Reducing partitions before write |

In [ ]:
# ── Checking and adjusting partitions ─────────────────────────────────────
df_base = make_large_df()

print(f"Default partitions: {df_base.rdd.getNumPartitions()}")

# repartition – increases or decreases with full shuffle
df_repartitioned = df_base.repartition(16)
print(f"After repartition(16): {df_repartitioned.rdd.getNumPartitions()}")

# coalesce – reduces only, no full shuffle
df_coalesced = df_repartitioned.coalesce(4)
print(f"After coalesce(4):     {df_coalesced.rdd.getNumPartitions()}")

# repartition by column – useful for filtering workloads
df_by_col = df_base.repartition(10, "category")
print(f"After repartition by column: {df_by_col.rdd.getNumPartitions()}")

In [ ]:
# ── Timed example: too few vs optimal vs too many partitions ──────────────
NUM_CORES = spark.sparkContext.defaultParallelism
print(f"Available cores: {NUM_CORES}")

results = []

for n_parts, label in [
    (1,           "Too few  (1 partition)"),
    (NUM_CORES,   f"Optimal  ({NUM_CORES} partitions = 1 per core)"),
    (NUM_CORES*4, f"Good     ({NUM_CORES*4} partitions = 4× cores)"),
    (200,         "Too many (200 partitions)"),
]:
    df_t = make_large_df().repartition(n_parts)
    t0 = time.time()
    df_t.groupBy("category").agg(
        F.avg("value").alias("avg"),
        F.sum("score").alias("total")
    ).count()
    elapsed = time.time() - t0
    results.append((label, n_parts, elapsed))
    print(f"{label:45s}  {elapsed:.2f}s")

print("\n(Expected: extreme ends slower; optimal range is 1–4× cores)")

In [ ]:
# ── Partition by column for efficient filtering ───────────────────────────
import tempfile, os

tmp_dir = tempfile.mkdtemp()
parquet_path = os.path.join(tmp_dir, "partitioned_data")

df_write = (
    spark.range(500_000)
    .withColumn("city",  F.element_at(
        F.array(*[F.lit(c) for c in ["NYC","LA","Chicago","Houston","Phoenix"]]),
        (F.col("id") % 5 + 1).cast("int")
    ))
    .withColumn("sales", F.rand(seed=1) * 10000)
)

df_write.write.partitionBy("city").mode("overwrite").parquet(parquet_path)
print(f"Written partitioned Parquet to: {parquet_path}")
print("Directory layout (city= folders):")
for entry in sorted(os.listdir(parquet_path)):
    print(f"  {entry}")

# Read back with partition pruning
t0 = time.time()
df_read_all = spark.read.parquet(parquet_path)
count_all = df_read_all.count()
t_all = time.time() - t0

t0 = time.time()
df_read_nyc = spark.read.parquet(parquet_path).filter(F.col("city") == "NYC")
count_nyc = df_read_nyc.count()
t_nyc = time.time() - t0

print(f"\nRead all cities ({count_all:,} rows): {t_all:.2f}s")
print(f"Read NYC only  ({count_nyc:,} rows): {t_nyc:.2f}s")
print("\n(Partition pruning reads only the NYC= subfolder — faster!)")

In [ ]:
# ── Detecting data skew ────────────────────────────────────────────────────
# Skew: one category has far more rows than others
from pyspark.sql import Row

# Create a skewed DataFrame: category 'A' has 90% of the data
n = 200_000
data_skewed = (
    [("A", float(i % 100)) for i in range(int(n * 0.9))] +
    [("B", float(i % 100)) for i in range(int(n * 0.05))] +
    [("C", float(i % 100)) for i in range(int(n * 0.05))]
)
df_skewed = spark.createDataFrame(data_skewed, ["category", "value"])

print("Partition row counts (skewed data):")
df_skewed.repartition(4, "category") \
    .groupBy(F.spark_partition_id().alias("partition_id")) \
    .count() \
    .orderBy("partition_id") \
    .show()

print("Category distribution:")
df_skewed.groupBy("category").count().orderBy("count", ascending=False).show()
print("→ Category A holds 90% of rows. Tasks processing it take 18× longer.")
print("→ Fix: add a random salt to the join/group key to spread the load.")

<h2 style="color:#E25822;">Part 4 – Broadcast Joins</h2>

### 4.1 Join Types and Their Costs

| Join strategy | When used | Shuffle? | Cost |
|---|---|:---:|---|
| **Broadcast hash join** | One side fits in driver memory | No | Very low |
| **Sort-merge join** | Both sides large | Yes | High |
| **Shuffle hash join** | Moderate sizes | Yes | Medium-High |

### 4.2 When to Broadcast

Use a broadcast join when:
- The **small** table is < `spark.sql.autoBroadcastJoinThreshold` (default **10 MB**)
- You are joining a **lookup / dimension table** to a large fact table

```
Without broadcast:            With broadcast:
Large  ─┐                     Large  ─────────────→ join (local)
         ├─ SHUFFLE ─→ join   Small  ─→ broadcast
Small  ─┘                             to every node
(expensive network I/O)       (no shuffle needed!)
```

### 4.3 autoBroadcastJoinThreshold

```python
# Raise threshold to 50 MB:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)

# Disable auto-broadcast:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
```

In [ ]:
# ── Timed example: regular join vs broadcast join ─────────────────────────
N_LARGE = 2_000_000
N_SMALL = 50          # lookup table – tiny

# Large fact table
df_large = (
    spark.range(N_LARGE)
    .withColumn("product_id", (F.col("id") % N_SMALL).cast("int"))
    .withColumn("sales",      F.rand(seed=42) * 500)
)

# Small dimension / lookup table
product_data = [(i, f"Product_{i}", float(i * 1.5)) for i in range(N_SMALL)]
df_small = spark.createDataFrame(product_data, ["product_id", "product_name", "price"])

# Disable auto-broadcast so Spark won't optimise automatically
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Regular (sort-merge) join
t0 = time.time()
result_regular = df_large.join(df_small, on="product_id", how="left")
cnt_regular    = result_regular.count()
time_regular   = time.time() - t0

# Broadcast join
t0 = time.time()
result_broadcast = df_large.join(broadcast(df_small), on="product_id", how="left")
cnt_broadcast    = result_broadcast.count()
time_broadcast   = time.time() - t0

# Re-enable auto-broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

print(f"Regular join   : {time_regular:.2f}s  (rows: {cnt_regular:,})")
print(f"Broadcast join : {time_broadcast:.2f}s  (rows: {cnt_broadcast:,})")
if time_broadcast > 0:
    print(f"Speedup        : {time_regular / time_broadcast:.1f}×")
print("\n(Expected: broadcast join 2–10× faster by eliminating the shuffle)")

In [ ]:
# ── spark.broadcast() for plain Python variables ──────────────────────────
# Useful when a Python dict/list is referenced inside a UDF on each executor

lookup_dict = {i: f"Product_{i}" for i in range(N_SMALL)}
bc_lookup   = spark.sparkContext.broadcast(lookup_dict)

# Use inside a UDF – the dict is sent once to each executor, not per row
from pyspark.sql.functions import udf

@udf(StringType())
def lookup_name(product_id):
    return bc_lookup.value.get(product_id, "Unknown")

df_with_name = df_large.withColumn("product_name", lookup_name(F.col("product_id").cast("int")))
df_with_name.select("product_id", "product_name").show(5, truncate=False)

# Always destroy broadcast variables when done to free memory
bc_lookup.destroy()
print("Broadcast variable destroyed.")

# Show join plans
print("\n── Regular join plan (SortMergeJoin): ──")
df_large.join(df_small, on="product_id").explain()
print("\n── Broadcast join plan (BroadcastHashJoin): ──")
df_large.join(broadcast(df_small), on="product_id").explain()

<h2 style="color:#E25822;">Part 5 – Query Optimization & Catalyst</h2>

### 5.1 The Catalyst Optimizer

Catalyst is Spark SQL's query optimiser.  It transforms your logical query plan through four phases:

```
1. Analysis        →  Resolve column names, types
2. Logical Opt.    →  Predicate pushdown, column pruning, constant folding
3. Physical Plan   →  Choose join strategies, sort algorithms
4. Code Gen.       →  Generate JVM bytecode (Project Tungsten)
```

You do not need to manage Catalyst manually — but understanding it helps you write queries that let it optimise effectively.

### 5.2 Reading explain() Output

```
df.explain()           → Physical plan only (most common)
df.explain("extended") → Parsed + Analysed + Optimised + Physical
df.explain("cost")     → Physical plan with cost estimates
df.explain("codegen")  → Generated Java code
```

Read the plan **bottom-up**: the leaf nodes are the data sources, the root is the final output.

### 5.3 Common Automatic Optimisations

| Optimisation | What it does | Example |
|---|---|---|
| **Predicate pushdown** | Moves filters to data source | Read only matching rows from Parquet |
| **Column pruning** | Drops unreferenced columns early | Don't read columns not in `select()` |
| **Constant folding** | Evaluates constant expressions at plan time | `1 + 2` → `3` before execution |
| **Join reordering** | Puts smaller tables on the build side | Better hash join performance |

### 5.4 Avoid Python UDFs — Use Built-in Functions

Python UDFs break Catalyst's code generation and require row-by-row serialisation (Python ↔ JVM).  Prefer:

1. **Built-in Spark functions** (`F.when`, `F.regexp_replace`, etc.) — fastest
2. **`pandas_udf`** (vectorised UDF) — 10–100× faster than regular UDF
3. **Regular Python UDF** — last resort

In [ ]:
# ── Predicate pushdown and column pruning ─────────────────────────────────
# Write a Parquet file so we can see pushdown in the plan
parquet_path2 = os.path.join(tmp_dir, "catalog")

(
    spark.range(500_000)
    .withColumn("dept",   (F.col("id") % 5).cast(StringType()))
    .withColumn("salary", F.rand(seed=99) * 80000 + 30000)
    .withColumn("age",    (F.rand(seed=77) * 40 + 22).cast("int"))
    .withColumn("name",   F.concat(F.lit("emp_"), F.col("id").cast(StringType())))
    .write.mode("overwrite").parquet(parquet_path2)
)

df_parquet = spark.read.parquet(parquet_path2)

# Bad: read all columns, then filter and select
print("=" * 60)
print("BEFORE optimisation (Spark still applies pushdown automatically):")
print("=" * 60)
df_parquet.filter(F.col("age") > 40).select("id", "dept", "salary").explain()

print("=" * 60)
print("SAME query – note PushedFilters and column selection in plan:")
print("=" * 60)
df_parquet.select("id", "dept", "salary", "age").filter(F.col("age") > 40) \
    .select("id", "dept", "salary").explain()

print("→ Look for 'PushedFilters' and 'ReadSchema' in the plan.")
print("→ Catalyst reads only the needed columns from Parquet (column pruning).")

In [ ]:
# ── UDF vs built-in function performance ──────────────────────────────────
try:
    import pandas as pd
    from pyspark.sql.functions import pandas_udf
    PANDAS_UDF_AVAILABLE = True
except ImportError:
    PANDAS_UDF_AVAILABLE = False

N_UDF = 1_000_000
df_udf_test = (
    spark.range(N_UDF)
    .withColumn("value", F.rand(seed=5) * 100)
)

# Regular Python UDF
from pyspark.sql.functions import udf

@udf(DoubleType())
def python_udf_double(x):
    return float(x) * 2.0

t0 = time.time()
df_udf_test.withColumn("doubled", python_udf_double(F.col("value"))).count()
t_python_udf = time.time() - t0

# Built-in Spark expression
t0 = time.time()
df_udf_test.withColumn("doubled", F.col("value") * 2.0).count()
t_builtin = time.time() - t0

print(f"Python UDF         : {t_python_udf:.2f}s")
print(f"Built-in expression: {t_builtin:.2f}s")
if t_builtin > 0:
    print(f"Speedup            : {t_python_udf / t_builtin:.1f}×")

# Pandas UDF (vectorised)
if PANDAS_UDF_AVAILABLE:
    @pandas_udf(DoubleType())
    def pandas_udf_double(s: pd.Series) -> pd.Series:
        return s * 2.0

    t0 = time.time()
    df_udf_test.withColumn("doubled", pandas_udf_double(F.col("value"))).count()
    t_pandas_udf = time.time() - t0
    print(f"Pandas UDF (vect.) : {t_pandas_udf:.2f}s")

print("\n(Expected: built-in ≈ 5–20× faster than Python UDF)")

<h2 style="color:#E25822;">Part 6 – PySpark vs Pandas Benchmark</h2>

How large does data need to be before PySpark becomes worthwhile?  We will measure two common workloads on 10 million rows.

> **Note on Colab:** Colab free tier has 2 virtual CPUs.  For larger cluster sizes the crossover point shifts toward pandas being competitive at even larger dataset sizes.  The benchmark here shows the local-mode overhead.

In [ ]:
# ── Generate 10M row dataset ──────────────────────────────────────────────
BM_ROWS = 10_000_000
print(f"Generating {BM_ROWS:,} row dataset ...")

# ── PySpark version ────────────────────────────────────────────────────────
CATEGORIES = ["Electronics", "Clothing", "Food", "Sports", "Books"]

t0 = time.time()
df_bm = (
    spark.range(BM_ROWS)
    .withColumn("category",
        F.element_at(
            F.array(*[F.lit(c) for c in CATEGORIES]),
            (F.col("id") % len(CATEGORIES) + 1).cast("int")
        )
    )
    .withColumn("value",  F.rand(seed=11) * 1000)
    .withColumn("value2", F.randn(seed=22) * 50 + 200)
    .withColumn("flag",   (F.rand(seed=33) > 0.5).cast("integer"))
)
df_bm.cache()
_ = df_bm.count()   # populate cache
t_spark_gen = time.time() - t0

# ── Pandas version ─────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
t0 = time.time()
df_pd = pd.DataFrame({
    "id":       np.arange(BM_ROWS),
    "category": rng.choice(CATEGORIES, size=BM_ROWS),
    "value":    rng.uniform(0, 1000, BM_ROWS),
    "value2":   rng.normal(200, 50, BM_ROWS),
    "flag":     rng.integers(0, 2, BM_ROWS),
})
t_pd_gen = time.time() - t0

print(f"PySpark data generated + cached : {t_spark_gen:.2f}s")
print(f"Pandas  data generated          : {t_pd_gen:.2f}s")
print(f"DataFrame size (pandas)         : {df_pd.memory_usage(deep=True).sum() / 1e6:.0f} MB")

In [ ]:
# ── Benchmark 1: groupBy aggregation ──────────────────────────────────────
print("Benchmark 1: groupBy + multi-aggregation")
print("-" * 50)

# PySpark
t0 = time.time()
result_spark = df_bm.groupBy("category").agg(
    F.count("id").alias("cnt"),
    F.avg("value").alias("avg_val"),
    F.sum("value2").alias("sum_val2"),
    F.max("value").alias("max_val"),
)
result_spark.collect()    # force full execution
t_spark_gb = time.time() - t0

# Pandas
t0 = time.time()
result_pd = df_pd.groupby("category").agg(
    cnt=("id",     "count"),
    avg_val=("value",  "mean"),
    sum_val2=("value2", "sum"),
    max_val=("value",  "max"),
)
_ = result_pd.shape     # force execution
t_pd_gb = time.time() - t0

print(f"PySpark groupBy  : {t_spark_gb:.2f}s")
print(f"Pandas  groupBy  : {t_pd_gb:.2f}s")
winner_gb = "Pandas" if t_pd_gb < t_spark_gb else "PySpark"
ratio_gb  = max(t_spark_gb, t_pd_gb) / min(t_spark_gb, t_pd_gb)
print(f"Winner: {winner_gb}  ({ratio_gb:.1f}×)")

In [ ]:
# ── Benchmark 2: filter + sort ─────────────────────────────────────────────
print("Benchmark 2: filter + sort")
print("-" * 50)

# PySpark
t0 = time.time()
result_spark2 = (
    df_bm
    .filter((F.col("value") > 500) & (F.col("flag") == 1))
    .orderBy("value", ascending=False)
)
cnt2 = result_spark2.count()
t_spark_fs = time.time() - t0

# Pandas
t0 = time.time()
result_pd2 = (
    df_pd[(df_pd["value"] > 500) & (df_pd["flag"] == 1)]
    .sort_values("value", ascending=False)
)
_ = len(result_pd2)
t_pd_fs = time.time() - t0

print(f"PySpark filter+sort : {t_spark_fs:.2f}s  (rows: {cnt2:,})")
print(f"Pandas  filter+sort : {t_pd_fs:.2f}s  (rows: {len(result_pd2):,})")
winner_fs = "Pandas" if t_pd_fs < t_spark_fs else "PySpark"
ratio_fs  = max(t_spark_fs, t_pd_fs) / min(t_spark_fs, t_pd_fs)
print(f"Winner: {winner_fs}  ({ratio_fs:.1f}×)")

In [ ]:
# ── Results summary table ──────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"{'Benchmark':30s} {'PySpark':>10} {'Pandas':>10} {'Winner':>12}")
print("=" * 65)
for label, t_sp, t_pd in [
    ("groupBy aggregation", t_spark_gb, t_pd_gb),
    ("filter + sort",       t_spark_fs, t_pd_fs),
]:
    winner = "Pandas" if t_pd < t_sp else "PySpark"
    print(f"{label:30s} {t_sp:>9.2f}s {t_pd:>9.2f}s {winner:>12}")
print("=" * 65)
print()
print("Decision guide (10M rows, local Spark):")
print("  • Pandas is often faster in local mode because there is no")
print("    cluster overhead and data fits in RAM.")
print("  • PySpark wins on a real multi-node cluster with 100M+ rows")
print("    where parallelism outweighs setup cost.")
print()
print("Rule of thumb:")
print("  < 1 GB  → pandas")
print("  1–10 GB → benchmark both")
print("  > 10 GB → PySpark (especially on a cluster)")

# Free memory
df_bm.unpersist()
del df_pd

<h2 style="color:#E25822;">Part 7 – Best Practices Checklist</h2>

The table below summarises the most impactful practices.  Code examples follow.

| # | Practice | Why it matters |
|---|---|---|
| 1 | **Avoid `collect()` on large data** | Pulls all data to driver → OOM |
| 2 | **Filter early (predicate pushdown)** | Reduces data read from disk |
| 3 | **Prefer DataFrames over RDDs** | Catalyst optimises DFs, not RDDs |
| 4 | **Use Parquet format** | Columnar, compressed, splittable |
| 5 | **Minimise shuffles** | Shuffles are the #1 performance killer |
| 6 | **Cache wisely** | Only cache if reused; always unpersist |
| 7 | **Monitor with Spark UI** | Identify bottlenecks in real time |

In [ ]:
# ── 1. Avoid collect() on large datasets ──────────────────────────────────
df_bp = spark.range(1_000_000).withColumn("val", F.rand())

# BAD: brings 1M rows to driver
# rows = df_bp.collect()   # DO NOT do this on large data

# GOOD: aggregation stays in the cluster, only the summary comes to driver
summary = df_bp.agg(F.avg("val"), F.max("val")).collect()
print("Collected summary only:", summary)

# GOOD: use show() to preview
df_bp.show(5)

# GOOD: use limit() then collect() when you truly need a small sample
sample = df_bp.limit(100).collect()
print(f"Sample size: {len(sample)} rows")

In [ ]:
# ── 2. Filter early ────────────────────────────────────────────────────────
df_fe = (
    spark.range(2_000_000)
    .withColumn("region",  (F.col("id") % 10).cast(StringType()))
    .withColumn("revenue", F.rand(seed=7) * 50000)
)

# BAD order: join first, then filter (processes 2M × join result)
# GOOD order: filter first, then join (processes ~200K × join result)

t0 = time.time()
result_late = (
    df_fe
    .withColumn("category", (F.col("id") % 5).cast(StringType()))
    .filter(F.col("region") == "3")      # filter AFTER expensive withColumn
    .agg(F.sum("revenue"))
    .collect()
)
t_late = time.time() - t0

t0 = time.time()
result_early = (
    df_fe
    .filter(F.col("region") == "3")      # filter BEFORE expensive withColumn
    .withColumn("category", (F.col("id") % 5).cast(StringType()))
    .agg(F.sum("revenue"))
    .collect()
)
t_early = time.time() - t0

print(f"Filter late  : {t_late:.3f}s")
print(f"Filter early : {t_early:.3f}s")
print("(Catalyst often reorders automatically, but always write filters first!)")

In [ ]:
# ── 3. DataFrames > RDDs ───────────────────────────────────────────────────
# DataFrames use Catalyst; RDDs use raw Python lambdas (no JVM optimisation)
N_COMPARE = 500_000
df_compare = spark.range(N_COMPARE).withColumn("val", F.rand())

t0 = time.time()
rdd_result = df_compare.rdd.map(lambda row: row["val"] * 2).filter(lambda v: v > 1.0).count()
t_rdd = time.time() - t0

t0 = time.time()
df_result = df_compare.withColumn("doubled", F.col("val") * 2).filter(F.col("doubled") > 1.0).count()
t_df = time.time() - t0

print(f"RDD approach        : {t_rdd:.2f}s  (result: {rdd_result:,})")
print(f"DataFrame approach  : {t_df:.2f}s  (result: {df_result:,})")

# ── 4. Parquet format ──────────────────────────────────────────────────────
parquet_path3 = os.path.join(tmp_dir, "fmt_test")
csv_path3     = os.path.join(tmp_dir, "fmt_test_csv")

df_fmt = spark.range(200_000).withColumn("val", F.rand())

t0 = time.time()
df_fmt.write.mode("overwrite").csv(csv_path3, header=True)
t_csv_write = time.time() - t0

t0 = time.time()
df_fmt.write.mode("overwrite").parquet(parquet_path3)
t_pq_write = time.time() - t0

# Approximate sizes
import glob
csv_size = sum(os.path.getsize(f) for f in glob.glob(f"{csv_path3}/*.csv"))
pq_size  = sum(os.path.getsize(f) for f in glob.glob(f"{parquet_path3}/*.parquet"))

print(f"\nCSV write    : {t_csv_write:.2f}s  |  size: {csv_size/1e6:.1f} MB")
print(f"Parquet write: {t_pq_write:.2f}s  |  size: {pq_size/1e6:.1f} MB")
print(f"Compression ratio: {csv_size/pq_size:.1f}× (Parquet is smaller!)")

In [ ]:
# ── 7. Monitor with Spark UI ───────────────────────────────────────────────
print("Spark UI access:")
print(f"  Local mode  : {spark.sparkContext.uiWebUrl}")
print("  Colab trick : Use a tunnel or check the output above")
print()
print("Spark UI tabs to check:")
print("  Jobs      → which jobs ran, duration, failures")
print("  Stages    → stage timeline, task distribution")
print("  Storage   → cached DataFrames and memory usage")
print("  SQL/DF    → query plan visualisation (most useful!)")
print("  Executors → memory, CPU, GC time per executor")
print()
print("Tip: A long 'shuffle read' or skewed task timeline = partition problem")
print("Tip: High GC time = memory pressure → tune executor.memory or persist level")
print()

# Trigger a job so you can inspect it in the UI
(
    spark.range(500_000)
    .withColumn("v", F.rand())
    .groupBy((F.col("id") % 10).alias("grp"))
    .agg(F.avg("v").alias("avg_v"))
    .orderBy("avg_v")
    .show(5)
)
print(f"\n→ Navigate to {spark.sparkContext.uiWebUrl} to inspect the job above.")

<h2 style="color:#E25822;">SparkSession Configuration Tips</h2>

Key configuration parameters and their effect on performance:

| Parameter | Default | Tuning advice |
|---|---|---|
| `spark.driver.memory` | 1g | Increase if driver OOMs (collect, broadcast) — try 4g |
| `spark.executor.memory` | 1g | Increase for memory-intensive jobs — start at 4g per executor |
| `spark.executor.cores` | 1 | 2–5 cores per executor is a common sweet spot |
| `spark.sql.shuffle.partitions` | 200 | **Most important**: set to 2–4× your total cores; 200 is too high for small clusters |
| `spark.default.parallelism` | #cores×2 | Controls RDD parallelism; match to `shuffle.partitions` |
| `spark.memory.fraction` | 0.6 | Fraction of executor memory for Spark execution; rest is user memory |
| `spark.sql.autoBroadcastJoinThreshold` | 10MB | Raise to 50–100 MB on large clusters with ample driver memory |

> **Rule:** `spark.sql.shuffle.partitions` is the single most common misconfiguration.  On a 4-core laptop, 200 partitions means 200 tiny tasks with massive overhead.  Set it to 8–16 for local mode.

In [ ]:
# ── Tuned SparkSession configuration template ─────────────────────────────
# This is a template — adjust based on your actual cluster resources

def create_optimised_session(
    app_name       = "MyApp",
    driver_memory  = "4g",
    executor_memory= "8g",
    executor_cores = 4,
    shuffle_parts  = 200,          # tune per cluster: 2-4× total cores
    broadcast_mb   = 50,
):
    """Return a tuned SparkSession.  Stop any existing session first."""
    SparkSession.builder.getOrCreate().stop()
    return (
        SparkSession.builder
        .master("local[*]")
        .appName(app_name)
        # Memory
        .config("spark.driver.memory",                         driver_memory)
        .config("spark.executor.memory",                       executor_memory)
        .config("spark.executor.cores",                        str(executor_cores))
        # Parallelism
        .config("spark.sql.shuffle.partitions",                str(shuffle_parts))
        .config("spark.default.parallelism",                   str(shuffle_parts))
        # Join optimisation
        .config("spark.sql.autoBroadcastJoinThreshold",        str(broadcast_mb * 1024 * 1024))
        # Adaptive Query Execution (Spark 3+)
        .config("spark.sql.adaptive.enabled",                  "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled","true")
        # Serialisation
        .config("spark.serializer",                            "org.apache.spark.serializer.KryoSerializer")
        # UI
        .config("spark.ui.showConsoleProgress",                "false")
        .getOrCreate()
    )

print("create_optimised_session() defined.")
print("Calling it now with local mode settings ...")
spark = create_optimised_session(
    shuffle_parts  = 8,
    driver_memory  = "4g",
    executor_memory= "4g",
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} restarted.")
print(f"shuffle.partitions  : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"autoBroadcastThresh : {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))//1024//1024} MB")
print(f"AQE enabled         : {spark.conf.get('spark.sql.adaptive.enabled')}")

In [ ]:
# ── Effect of spark.sql.shuffle.partitions on a groupBy ───────────────────
df_sp = (
    spark.range(1_000_000)
    .withColumn("cat", (F.col("id") % 20).cast(StringType()))
    .withColumn("val", F.rand())
)

print("Effect of spark.sql.shuffle.partitions on a groupBy:")
print("-" * 55)

for n_parts in [2, 8, 50, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", str(n_parts))
    t0 = time.time()
    df_sp.groupBy("cat").agg(F.avg("val")).count()
    elapsed = time.time() - t0
    print(f"  shuffle.partitions={n_parts:>3}  →  {elapsed:.2f}s")

# Reset to a good value
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("\n(Too low = underutilised cores; too high = scheduling overhead)")

<h2 style="color:#E25822;">Summary – Performance Optimization Cheat Sheet</h2>

| Topic | Key Concept | Quick Action |
|---|---|---|
| **Execution model** | Jobs → Stages → Tasks; DAG is lazy | Call `.explain()` to inspect the plan |
| **Narrow vs wide** | Narrow = no shuffle; wide = shuffle | Minimise groupBy/join shuffles |
| **Caching** | Cache only if reused > once | `df.cache()` then `.count()` to populate; `.unpersist()` when done |
| **Storage levels** | MEMORY_ONLY fastest; MEMORY_AND_DISK safer | Default `cache()` = MEMORY_AND_DISK |
| **Partitions** | 2–4× cores; 128–512 MB per partition | `repartition()` to increase; `coalesce()` to decrease |
| **Partition by col** | Efficient filter on written data | `df.write.partitionBy("col")` |
| **Data skew** | One key dominates → slow straggler tasks | Salt the skewed key or use AQE |
| **Broadcast join** | No shuffle when one table is small | `df.join(broadcast(small_df), ...)` |
| **Broadcast var** | Python dict/list sent once to executors | `sc.broadcast(dict)` |
| **Catalyst** | Auto predicate pushdown + column pruning | Write filters early; use built-in functions |
| **UDFs** | Break Catalyst; row-by-row Python↔JVM | Use `F.*` functions or `pandas_udf` |
| **collect()** | Brings ALL data to driver | Only collect summaries or small samples |
| **Parquet** | 5–10× smaller than CSV; columnar | Always write/read Parquet in production |
| **shuffle.partitions** | Default 200 is too high for small clusters | Set to 2–4× total cores |
| **AQE** | Spark 3+ auto-tunes at runtime | Enable with `spark.sql.adaptive.enabled=true` |
| **Spark UI** | Visual debugging of jobs and stages | `spark.sparkContext.uiWebUrl` |
| **PySpark vs pandas** | Spark wins only at scale (>1–10 GB) | Benchmark both before committing |

<h2 style="color:#E25822;">Exercises</h2>

Work through each exercise independently.  Reference cells in this notebook for guidance.

---

### Exercise 1 – Caching and Partition Tuning

1. Generate a Spark DataFrame with **8 million rows** containing columns: `id`, `product` (one of 20 product names), `region` (one of 8 regions), `revenue` (random float 0–10000), `quantity` (random integer 1–100).
2. Cache the DataFrame and confirm it is cached by printing its `storageLevel`.
3. Run the following **three aggregations** and measure the total time *without* cache (fresh DataFrame each time) and *with* cache:
   - Total revenue per `region`
   - Average quantity per `product`
   - Count of rows where `revenue > 7500` per `region`
4. Experiment with `spark.sql.shuffle.partitions` values of **2, 8, 32, 200** and record the time for aggregation 1 at each setting.  Which value is fastest on your machine?
5. Unpersist the DataFrame when done.

---

### Exercise 2 – Broadcast Join vs Sort-Merge Join

1. Create a **large** DataFrame of 3 million order rows with columns: `order_id`, `customer_id` (1–500), `amount` (random float).
2. Create a **small** lookup DataFrame of 500 customer rows with columns: `customer_id`, `customer_name`, `tier` (Gold/Silver/Bronze).
3. Disable auto-broadcast by setting `spark.sql.autoBroadcastJoinThreshold = -1`.
4. Time a **regular join** (orders ⟕ customers on `customer_id`).  Print the query plan and identify the join strategy used.
5. Time a **broadcast join** using the `broadcast()` hint.  Print the plan and identify the strategy.
6. Re-enable auto-broadcast and set the threshold to **1 MB**.  Run the join *without* the hint — does Spark automatically choose the broadcast strategy now?  Why?
7. Report the speedup and explain in 2–3 sentences *why* the broadcast join is faster.

---

### Exercise 3 – UDF Performance and Catalyst Explain

1. Create a Parquet file with **2 million rows**: `id`, `department` (one of 6 departments), `salary` (random float 30000–120000), `years_exp` (random integer 1–30).
2. Implement the following transformation in **three ways** and time each:
   - **Python UDF:** classify salary as `'Low'` (<50k), `'Mid'` (50k–90k), or `'High'` (>90k)
   - **Built-in `F.when()`:** same classification
   - **`pandas_udf`:** same classification
3. For the built-in approach, write a query that filters `department == 'Engineering'` and selects only `id`, `salary`, `salary_band` (the new column).  Run `df.explain()` and identify:
   - Where **predicate pushdown** occurs (look for `PushedFilters`)
   - Which columns are read from Parquet (look for `ReadSchema`)
4. Report your timing results in a formatted table and conclude: when would you use each approach?

In [ ]:
# ── Exercise 1 scaffold ────────────────────────────────────────────────────
# TODO: replace the placeholders below with your implementation

PRODUCTS = [f"Product_{i}" for i in range(20)]
REGIONS  = [f"Region_{i}" for i in range(8)]

# 1. Generate DataFrame
df_ex1 = (
    spark.range(8_000_000)
    # TODO: add product, region, revenue, quantity columns
)

# 2. Cache and verify
# TODO: cache df_ex1, print storageLevel

# 3. Time aggregations with and without cache
# TODO: implement timing loop

# 4. Tune shuffle partitions
# TODO: loop over [2, 8, 32, 200] and record timing

# 5. Unpersist
# TODO: unpersist df_ex1

print("Exercise 1 scaffold loaded – implement the TODOs above.")

In [ ]:
# ── Exercise 2 scaffold ────────────────────────────────────────────────────
TIERS = ["Gold", "Silver", "Bronze"]

# 1. Large orders DataFrame
df_orders = (
    spark.range(3_000_000)
    # TODO: add customer_id, amount columns
)

# 2. Small customers lookup
# TODO: create df_customers with 500 rows

# 3. Disable auto-broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# 4. Regular join + timing
# TODO: time the regular join, show explain()

# 5. Broadcast join + timing
# TODO: time the broadcast join, show explain()

# 6. Re-enable at 1 MB and observe
# TODO: set threshold to 1MB, run join without hint, check plan

# Re-enable default threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print("Exercise 2 scaffold loaded – implement the TODOs above.")

In [ ]:
# ── Exercise 3 scaffold ────────────────────────────────────────────────────
DEPTS = ["Engineering", "Marketing", "Finance", "HR", "Sales", "Operations"]
parquet_ex3 = os.path.join(tmp_dir, "exercise3")

# 1. Generate and write Parquet
# TODO: create 2M row DataFrame and write as Parquet

# 2a. Python UDF classification
# TODO: define python_udf_band, time withColumn + count()

# 2b. Built-in F.when() classification
# TODO: implement using F.when().when().otherwise(), time it

# 2c. Pandas UDF classification
# TODO: implement @pandas_udf, time it

# 3. explain() analysis
# TODO: read parquet, filter dept==Engineering, select id+salary+salary_band, explain()

# 4. Results table
# TODO: print formatted comparison table

print("Exercise 3 scaffold loaded – implement the TODOs above.")

In [ ]:
# ── Cleanup ────────────────────────────────────────────────────────────────
import shutil

shutil.rmtree(tmp_dir, ignore_errors=True)
print(f"Temporary files removed: {tmp_dir}")

spark.stop()
print("SparkSession stopped. Notebook complete.")